<a href="https://colab.research.google.com/github/dev-gauravpingale/banking-data-platform-pyspark/blob/main/notebooks/03_Bronze_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bronze Layer

## Objective

Build the Bronze Layer of the Banking Data Platform.

Responsibilities:

- Read raw CSV files
- Apply explicit schema
- Preserve raw data
- Add ingestion metadata
- Write Parquet files

No business transformations are performed in Bronze.

In [95]:
#install pyspark if not installed

In [96]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import*

In [97]:
spark = SparkSession.builder.appName("Banking Bronze Layer").getOrCreate()

In [98]:
spark.version

'4.0.3'

In [99]:
PROJECT_ROOT = "/content/banking-data-platform-pyspark"

LANDING_PATH = f"{PROJECT_ROOT}/data/landing"
BRONZE_PATH = f"{PROJECT_ROOT}/data/bronze"
SILVER_PATH = f"{PROJECT_ROOT}/data/silver"
GOLD_PATH = f"{PROJECT_ROOT}/data/gold"

In [100]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType
)

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("occupation", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("kyc_status", StringType(), True),
    StructField("risk_rating", StringType(), True),
    StructField("created_date", DateType(), True)
])

In [101]:
customers_df =(
    spark.read
    .option("header", "true")
    .schema(customer_schema)
    .csv(f"{LANDING_PATH}/customers.csv")
)

In [102]:
customers_df.show(5,False)

+-----------+----------+---------+------+-------------+-------------------------+----------+---------+-----------+-----------------+----------------+----------+-----------+------------+
|customer_id|first_name|last_name|gender|date_of_birth|email                    |phone     |city     |state      |occupation       |customer_segment|kyc_status|risk_rating|created_date|
+-----------+----------+---------+------+-------------+-------------------------+----------+---------+-----------+-----------------+----------------+----------+-----------+------------+
|CUST000001 |Daksh     |Bakshi   |Male  |1996-05-13   |daksh.bakshi760@gmail.com|1819600133|Mumbai   |Maharashtra|Business Owner   |Retail          |Completed |Low        |2022-09-26  |
|CUST000002 |Hardik    |Saini    |Male  |1971-11-16   |hardik.saini433@gmail.com|2654235116|Ahmedabad|Gujarat    |Software Engineer|Retail          |Completed |Low        |2024-06-16  |
|CUST000003 |Kevin     |Bassi    |Male  |1995-10-23   |kevin.bassi204@

In [103]:
print(f"Total Records : {customers_df.count()}")

Total Records : 500


In [104]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- risk_rating: string (nullable = true)
 |-- created_date: date (nullable = true)



In [105]:
customers_df.filter(col("customer_id").isNull()).show()

+-----------+----------+---------+------+-------------+-----+-----+----+-----+----------+----------------+----------+-----------+------------+
|customer_id|first_name|last_name|gender|date_of_birth|email|phone|city|state|occupation|customer_segment|kyc_status|risk_rating|created_date|
+-----------+----------+---------+------+-------------+-----+-----+----+-----+----------+----------------+----------+-----------+------------+
+-----------+----------+---------+------+-------------+-----+-----+----+-----+----------+----------------+----------+-----------+------------+



In [106]:
customers_df.groupBy("customer_id")\
    .count()\
    .filter(col("count") > 1)\
    .show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [107]:
print("="*50)
print("Customer Dataset Validation")
print("="*50)
print(f"Total Records : {customers_df.count()}")
print("Distinct Customers :",customers_df.select("customer_id").distinct().count())

Customer Dataset Validation
Total Records : 500
Distinct Customers : 500


In [108]:
customers_bronze_df = \
customers_df\
.withColumn("ingestion_timestamp", current_timestamp())\
.withColumn("source_file", lit("customers.csv"))

In [109]:
customers_bronze_df.show(5, truncate=False)

+-----------+----------+---------+------+-------------+-------------------------+----------+---------+-----------+-----------------+----------------+----------+-----------+------------+--------------------------+-------------+
|customer_id|first_name|last_name|gender|date_of_birth|email                    |phone     |city     |state      |occupation       |customer_segment|kyc_status|risk_rating|created_date|ingestion_timestamp       |source_file  |
+-----------+----------+---------+------+-------------+-------------------------+----------+---------+-----------+-----------------+----------------+----------+-----------+------------+--------------------------+-------------+
|CUST000001 |Daksh     |Bakshi   |Male  |1996-05-13   |daksh.bakshi760@gmail.com|1819600133|Mumbai   |Maharashtra|Business Owner   |Retail          |Completed |Low        |2022-09-26  |2026-07-08 19:12:55.037462|customers.csv|
|CUST000002 |Hardik    |Saini    |Male  |1971-11-16   |hardik.saini433@gmail.com|2654235116|

In [110]:
CUSTOMERS_BRONZE_PATH = f"{BRONZE_PATH}/customers"

In [111]:
customers_bronze_df.write.mode("overwrite").parquet(CUSTOMERS_BRONZE_PATH)

In [112]:
bronze_customers_df = spark.read.parquet(CUSTOMERS_BRONZE_PATH)
bronze_customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- risk_rating: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [113]:
bronze_customers_df.show(5,False)

+-----------+----------+---------+------+-------------+-------------------------+----------+---------+-----------+-----------------+----------------+----------+-----------+------------+--------------------------+-------------+
|customer_id|first_name|last_name|gender|date_of_birth|email                    |phone     |city     |state      |occupation       |customer_segment|kyc_status|risk_rating|created_date|ingestion_timestamp       |source_file  |
+-----------+----------+---------+------+-------------+-------------------------+----------+---------+-----------+-----------------+----------------+----------+-----------+------------+--------------------------+-------------+
|CUST000001 |Daksh     |Bakshi   |Male  |1996-05-13   |daksh.bakshi760@gmail.com|1819600133|Mumbai   |Maharashtra|Business Owner   |Retail          |Completed |Low        |2022-09-26  |2026-07-08 19:12:55.168287|customers.csv|
|CUST000002 |Hardik    |Saini    |Male  |1971-11-16   |hardik.saini433@gmail.com|2654235116|